In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from pandas.core.series import Series
from PIL import Image
import plotly.graph_objects as go


dataset_classes = sorted(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
			'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 
			'del', 'nothing', 'space'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_path = "../Data/asl_alphabet_train/asl_alphabet_train"
test_path = "../Data/asl_alphabet_test/asl_alphabet_test/"

IMAGE_SIZE = 64
BATCH_SIZE = 64
device

device(type='cuda')

In [ ]:


train_path_df = pd.read_csv("../Data/train_path_dataframe.csv")
# my_data_df = pd.read_csv("../Data2/my_data_path_dataframe.csv")

In [ ]:
# Training transforms with Augmentation (Bonus)
train_transforms = transforms.Compose([
	transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
	transforms.RandomRotation(10),           # Randomly rotate the image by up to 15 degrees
	transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue = 0.2),  # Randomly change brightness
	transforms.RandomGrayscale(p=0.2), # Randomly transform to black and white
	transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Randomly shift the image
	transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), # random blur
	transforms.ToTensor()
 	# transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # برای درگیر میزان نور نشدن 
])

# train_transforms = transforms.Compose([
#     # ۱. اندازه دست مهم نیست (تغییر مقیاس تصادفی برای بی‌تفاوت کردن مدل به کوچک/بزرگ بودن دست)
#     transforms.RandomResizedCrop(
#         IMAGE_SIZE, 
#         scale=(0.7, 1.0),      # دست می‌تواند ۷۰٪ تا ۱۰۰٪ کادر را پر کند
#         ratio=(0.9, 1.1)
#     ),
    
#     # ۲. جایگاه دست مهم نیست (انتقال، چرخش و زاویه تصادفی)
#     transforms.RandomRotation(15),
#     transforms.RandomAffine(degrees=0, translate=(0.15, 0.15)), # جابجایی دست در کادر
    
#     # ۳. رنگ مهم نیست (تبدیل مستقیم به Grayscale با ۳ کانال جهت سازگاری با CNN)
#     transforms.Grayscale(num_output_channels=3),
    
#     # ۴. میزان نور مهم نیست (تغییرات شدید کنتراست و روشنایی روی تصویر سیاه و سفید)
#     transforms.ColorJitter(brightness=0.5, contrast=0.5),
    
#     transforms.ToTensor(),
    
#     # ۵. استخراج بهتر لبه‌ها و خنثی‌سازی نوسانات کلی نور
#     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
# ])

# Validation/Test transforms (Clean data)
test_transforms = transforms.Compose([
	transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
	transforms.ToTensor()
 	# transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # برای درگیر میزان نور نشدن 
])

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(train_path_df['image_path'], train_path_df['label'], test_size = 0.2, random_state=54)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train,  Y_train, test_size = 0.25, random_state=54)

# X_test2, Y_test2 = my_data_df['image_path'], my_data_df['label']

In [ ]:
class ASLDataset(Dataset):
	def __init__(self, X: Series, Y: Series, transform=None):
		self.X = X.reset_index(drop = True)
		self.Y = Y.reset_index(drop = True)
		# self.dataframe = dataframe.reset_index(drop=True)
		self.transform = transform
		# Create a mapping from text labels (A, B, del...) to integers (0, 1, 2...)
		self.labels = sorted(self.Y.unique())
		self.idx = {label: idx for idx, label in enumerate(self.labels)}
	
	def __len__(self):
		return len(self.X)
	
	def __getitem__(self, idx):
		# Fetch image path and class string
		img_path = self.X[idx]
		class_name = self.Y[idx]
	       	
		       # Load the image lazily (only when requested)
		image = Image.open(img_path).convert('RGB')
		label_idx = self.idx[class_name]
		
		# Apply the resizing and normalization transforms
		if self.transform:
			image = self.transform(image)
					
		return image, torch.tensor(label_idx, dtype=torch.long)

# 3. Instantiate the Train, Validation, and Test Datasets
train_dataset = ASLDataset(X_train, Y_train, transform=train_transforms)
valid_dataset = ASLDataset(X_valid, Y_valid, transform=test_transforms)
test_dataset = ASLDataset(X_test, Y_test, transform=test_transforms)

# train2_dataset = ASLDataset(X_test2, Y_test2, transform = train_transforms)
# test2_dataset = ASLDataset(X_test2, Y_test2, transform = test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 9)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)

# train2_loader = DataLoader(train2_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)
# test2_loader = DataLoader(test2_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)

In [30]:
class CNN(nn.Module):
        def __init__(self, num_classes):
                super().__init__()
                self.POOL_SIZE = 2
                self.L1_OUT_CHANNEL  = 32
                self.L1_KERNEL_SIZE  = 3
                self.L2_OUT_CHANNEL  = 64
                self.L2_KERNEL_SIZE  = 3
                self.L3_OUT_CHANNEL  = 128
                self.L3_KERNEL_SIZE  = 3
                self.DROPOUT_P = 0.5
                self.L1_NEURON = 512
                self.FINAL_DIM = IMAGE_SIZE // 8 # x Layers --> 2 ** x
                
                
                self.pool = nn.MaxPool2d(
                        kernel_size = self.POOL_SIZE,
                        stride = self.POOL_SIZE
                )
                self.relu = nn.ReLU()
                
                # Layer 1
                
                self.conv1 = nn.Conv2d(
                        in_channels = 3,
                        out_channels = self.L1_OUT_CHANNEL,
                        kernel_size =self. L1_KERNEL_SIZE,
                        padding = (self.L1_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm1 = nn.BatchNorm2d(self.L1_OUT_CHANNEL)
                
                # Layer  2
                
                self.conv2 = nn.Conv2d(
                        in_channels = self.L1_OUT_CHANNEL,
                        out_channels = self.L2_OUT_CHANNEL,
                        kernel_size = self.L2_KERNEL_SIZE,
                        padding = (self.L2_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm2 = nn.BatchNorm2d(self.L2_OUT_CHANNEL)
                
                # Layer  3
                self.conv3 = nn.Conv2d(
                        in_channels = self.L2_OUT_CHANNEL,
                        out_channels = self.L3_OUT_CHANNEL,
                        kernel_size = self.L3_KERNEL_SIZE,
                        padding = (self.L3_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm3 = nn.BatchNorm2d(self.L3_OUT_CHANNEL)
                
                
                # Fully Connected (Dense) Layers
                self.dropout = nn.Dropout(self.DROPOUT_P)
                # After three MaxPools of 2x2, a 64x64 image becomes 8x8. 
                # So the flattened size is 128 channels * FINAL_DIM * FINAL_DIM
                
                self.fully_connected1 = nn.Linear(self.L3_OUT_CHANNEL * self.FINAL_DIM * self.FINAL_DIM, self.L1_NEURON)
                self.fully_connected2 = nn.Linear(self.L1_NEURON, num_classes)
                
        def forward(self, x: torch.Tensor):
                # L1
                x = self.pool(self.relu(self.batch_norm1(self.conv1(x))))
                
                # L2
                x = self.pool(self.relu(self.batch_norm2(self.conv2(x))))
                
                # L3
                x = self.pool(self.relu(self.batch_norm3(self.conv3(x))))
                
                x = x.view(-1, self.L3_OUT_CHANNEL * self.FINAL_DIM * self.FINAL_DIM)
                
                x = self.dropout(self.relu(self.fully_connected1(x)))
                x = self.fully_connected2(x)
                
                return x

In [31]:
def learning_process(
        model: nn.Module,
        train_loader: DataLoader = train_loader,
        valid_loader: DataLoader = valid_loader,
        criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss(),
        learning_rate = 0.0001,
        epochs = 13
        ):
        # optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        
        train_losses, valid_losses = [], []
        train_accs, valid_accs = [], []
        
        best_val_loss = float('inf') # متغیری برای ذخیره بهترین Loss
        
        # Training
        for epoch in range(epochs):
                print(f"------------ {epoch = } Started")
                # Set model to training mode (enables Dropout and BatchNorm updates)
                model.train()
                
                running_loss, correct, total = 0.0, 0, 0
                for i, (images, labels) in enumerate(train_loader):
                        images, labels = images.to(device), labels.to(device)
                        # Clear old gradients
                        optimizer.zero_grad()
                        # Forward pass: predict labels
                        outputs = model(images)
                        # Calculate error
                        loss = criterion(outputs, labels)
                        # Backward pass: compute gradients
                        loss.backward()
                        # Update weights
                        optimizer.step()
                        
                        running_loss += loss.item()
                        _, predicted = torch.max(outputs.data, 1)
                        total += labels.size(0)
                        correct += (predicted == labels).sum().item()
                        
                train_losses.append(running_loss / len(train_loader))
                train_accs.append(100 * correct / total)
                
                model.eval()
                valid_loss, correct, total = 0.0, 0, 0
                
                with torch.no_grad():
                        for images, labels in valid_loader:
                                images, labels = images.to(device), labels.to(device)
				# Forward pass: predict labels
                                outputs = model(images)
				# Calculate error
                                loss = criterion(outputs, labels)
				
                                valid_loss += loss.item()
                                _, predicted = torch.max(outputs, 1)
                                total += labels.size(0)
                                correct += (predicted == labels).sum().item()
                valid_losses.append(valid_loss / len(valid_loader))
                valid_accs.append(100 * correct / total)
                
                print(f"Epoch {epoch+1}/{epochs} | "
                      f"Train Loss: {train_losses[-1]:.4f}, Acc: {train_accs[-1]:.2f}% | "
                      f"Val Loss: {valid_losses[-1]:.4f}, Acc: {valid_accs[-1]:.2f}%")
                
                if valid_losses[-1] < best_val_loss:
                        best_val_loss = valid_losses[-1]
                        torch.save(model.state_dict(), 'best_sign_language_model.pth')
                        print(f"Best model saved with Val Loss: {best_val_loss:.4f}")
        return train_losses, train_accs, valid_losses, valid_accs

In [ ]:
model = CNN(num_classes=29).to(device)

train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, epochs=20)
# train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, valid_loader=test2_loader, epochs=20)
# train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, train_loader = train2_loader)

------------ epoch = 0 Started
Epoch 1/20 | Train Loss: 2.8335, Acc: 18.54% | Val Loss: 2.6345, Acc: 19.31%
Best model saved with Val Loss: 2.6345
------------ epoch = 1 Started
Epoch 2/20 | Train Loss: 2.0380, Acc: 37.88% | Val Loss: 2.2581, Acc: 31.79%
Best model saved with Val Loss: 2.2581
------------ epoch = 2 Started
Epoch 3/20 | Train Loss: 1.6697, Acc: 48.09% | Val Loss: 2.0814, Acc: 32.85%
Best model saved with Val Loss: 2.0814
------------ epoch = 3 Started
Epoch 4/20 | Train Loss: 1.4252, Acc: 55.05% | Val Loss: 2.0403, Acc: 36.69%
Best model saved with Val Loss: 2.0403
------------ epoch = 4 Started
Epoch 5/20 | Train Loss: 1.2562, Acc: 60.11% | Val Loss: 1.5516, Acc: 46.99%
Best model saved with Val Loss: 1.5516
------------ epoch = 5 Started
Epoch 6/20 | Train Loss: 1.1338, Acc: 63.87% | Val Loss: 1.9032, Acc: 40.45%
------------ epoch = 6 Started
Epoch 7/20 | Train Loss: 1.0275, Acc: 67.06% | Val Loss: 1.5125, Acc: 47.54%
Best model saved with Val Loss: 1.5125
----------

In [25]:
torch.save(model.state_dict(), 'last_sign_language_model.pth')

print("Model saved successfully!")

Model saved successfully!


In [19]:
loaded_model = CNN(num_classes=29).to(device)
loaded_model.load_state_dict(torch.load('best_sign_language_model.pth'))
loaded_model.eval()

CNN(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fully_connected1): Linear(in_features=8192, out_features=512, bias=True)
  (fully_connected2): Linear(in_features=512, out_features=29, bias=True)
)

In [36]:
def test_model(model, test_loader = test_loader):
	eval_model = model # or loaded_model

	# 2. Set the model to evaluation mode (Crucial: turns off Dropout and BatchNorm updates)
	eval_model.eval()

	# 3. Define the loss function (since it was previously trapped inside your learning_process function)
	criterion = nn.CrossEntropyLoss()

	test_loss = 0.0
	correct = 0
	total = 0

	# Lists to store all predictions and true labels for the Confusion Matrix
	all_preds = []
	all_true = []

	print("Starting evaluation on the Test Dataset...")

	# Turn off gradient calculation to save memory and speed up inference
	with torch.no_grad():
		for images, labels in test_loader:
			# Move data to the same device as the model (cuda or cpu)
			images, labels = images.to(device), labels.to(device)
			
			# Forward pass
			outputs = eval_model(images)
			loss = criterion(outputs, labels)
			
			# Track Loss and Accuracy
			test_loss += loss.item()
			_, predicted = torch.max(outputs.data, 1)
			
			total += labels.size(0)
			correct += (predicted == labels).sum().item()
			
			# Store predictions and true labels for the matrix
			all_preds.extend(predicted.cpu().numpy())
			all_true.extend(labels.cpu().numpy())

	# Calculate final averages
	avg_test_loss = test_loss / len(test_loader)
	test_accuracy = 100 * correct / total

	print("-" * 30)
	print(f"Final Test Loss: {avg_test_loss:.4f}")
	print(f"Final Test Accuracy: {test_accuracy:.2f}%")
	print("-" * 30)

	# 4. Generate and Plot the Confusion Matrix

	cm = confusion_matrix(all_true, all_preds)

	# Create custom colorscale with boundaries
	colors = ["#ffffff", "#d53737", "#3664a4", "#387d13"]
	boundaries = [0, 20, 300, 550, 999]

	# Normalize boundaries to 0-1 for colorscale
	min_val = min(boundaries)
	max_val = max(boundaries)
	normalized_boundaries = [(b - min_val) / (max_val - min_val) for b in boundaries]

	# Create colorscale
	colorscale = []
	for i in range(len(colors)):
		colorscale.append([normalized_boundaries[i], colors[i]])
		colorscale.append([normalized_boundaries[i+1], colors[i]])
	print(colorscale)
	# Create heatmap
	fig = go.Figure(data=go.Heatmap(
		z=cm,  # your confusion matrix
		x=dataset_classes,
		y=dataset_classes,
		colorscale=colorscale,
		zmin=min(boundaries),
		zmax=np.max(cm.sum(axis = 1)),
		text=cm,
		texttemplate='%{text}',
		textfont={"size": 10},
		hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
	))

	# Update layout
	fig.update_layout(
		title='Confusion Matrix: Test Dataset',
		xaxis_title='Predicted Label',
		yaxis_title='True Label',
		width=800,
		height=700,
		xaxis={'tickangle': 45}
	)

	fig.show()

In [21]:
test_model(loaded_model)
# test_model(loaded_model, test_loader = test2_loader)

Starting evaluation on the Test Dataset...
------------------------------
Final Test Loss: 0.0493
Final Test Accuracy: 98.33%
------------------------------
[[0.0, '#ffffff'], [0.02002002002002002, '#ffffff'], [0.02002002002002002, '#d53737'], [0.3003003003003003, '#d53737'], [0.3003003003003003, '#3664a4'], [0.5505505505505506, '#3664a4'], [0.5505505505505506, '#387d13'], [1.0, '#387d13']]


If your data is balanced but the model still fails, you need to force the model to pay more attention to the difficult class. You can do this by modifying your CrossEntropyLoss to weigh the errors differently.  The Logic: If the model gets a 'space' image wrong, we want the "fine" to be much higher than if it gets an 'A' image wrong.

In [ ]:
weights = [1.0] * 29
weights[dataset_classes.index('M')] = 2.0

class_weights = torch.tensor(weights) # Increase the weight for the difficult class
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

model_v2 = CNN(num_classes=29).to(device)
train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model_v2, criterion = criterion)
torch.save(model.state_dict(), 'sign_language_modelv2.pth')
test_model(model = model_v2)


In [ ]:
# test_model(model, test_loader = test2_loader)

Starting evaluation on the Test Dataset...
------------------------------
Final Test Loss: 1.4841
Final Test Accuracy: 52.95%
------------------------------
[[0.0, '#ffffff'], [0.02002002002002002, '#ffffff'], [0.02002002002002002, '#d53737'], [0.3003003003003003, '#d53737'], [0.3003003003003003, '#3664a4'], [0.5505505505505506, '#3664a4'], [0.5505505505505506, '#387d13'], [1.0, '#387d13']]


In [ ]:
# torch.save(loaded_model.state_dict(), 'my_sign_language_model.pth')

# print("Model saved successfully!")

Model saved successfully!
